# Linear Regression Lab — Predicting Network Latency

## Objective

Use linear regression to predict **Round-Trip Time (RTT)** from traceroute measurements:
- **Hop count** — number of routers traversed
- **Geographic distance** — physical distance between source and destination (km)
- **Time of day** — hour of measurement (affects congestion)

## Learning Goals

1. Understand the physical intuition behind linear regression in networking
2. Build a single-feature model (hops → RTT)
3. Build a multi-feature model (hops + distance + hour → RTT)
4. Evaluate and compare models using MSE, RMSE, and R²
5. Interpret model coefficients in networking terms

## 1. Setup and Imports

This lab relies on **scikit-learn** Python library([scikit-learn.org](https://scikit-learn.org/)) to train the machine-learning models, to split the training dataset, and compute evaluation metrics.

We also use four foundational Python libraries for scientific computing and data visualisation:

- **NumPy** ([numpy.org](https://numpy.org/)) — the core library for numerical computing in Python. It provides fast, memory-efficient N-dimensional arrays and vectorised mathematical operations that underpin nearly every data-science tool.
- **pandas** ([pandas.pydata.org](https://pandas.pydata.org/)) — a data-analysis library built on NumPy. Its `DataFrame` structure makes it easy to load, clean, filter, and aggregate tabular data such as our traceroute measurements.
- **Matplotlib** ([matplotlib.org](https://matplotlib.org/)) — the foundational plotting library for Python. It gives fine-grained control to create static charts (histograms, scatter plots, bar charts) for exploring and presenting data.
- **seaborn** ([seaborn.pydata.org](https://seaborn.pydata.org/)) — a statistical visualisation library built on top of Matplotlib. It offers a higher-level interface and attractive default styles for plots like correlation heatmaps and distribution charts.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
np.random.seed(42)

print("Libraries loaded successfully.")

## 2. Load the Traceroute Dataset

The first step in any machine learning project is familiarize yourself with the data. You'll use the Pandas library for this. The most important part of the Pandas library is the DataFrame. A DataFrame holds the type of data you might think of as a table. This is similar to a sheet in Excel, or a table in a SQL database.

Pandas has powerful methods for most things you'll want to do with this type of data.

We load 1,000 traceroute measurements from a CSV file. Each row contains:

- **hops** — number of routers traversed (3–24)
- **distance_km** — geographic distance between source and destination
- **hour** — hour of day the measurement was taken (0–23)
- **is_peak** — 1 if measured during business hours (9–18), else 0
- **rtt_ms** — measured Round-Trip Time (the target to predict)

In [ ]:
#df = pd.read_csv("traceroute_latency.csv")
url = "https://raw.githubusercontent.com/IntuitiveSoft-net/ml-datasets/main/traceroute_latency.csv"
df = pd.read_csv(url)

print(f"Dataset shape: {df.shape}")
df.head(10)

## 3. Exploratory Data Analysis (EDA)

The first step in any machine learning project is familiarize yourself with the data. You'll use the Pandas library for this. The most important part of the Pandas library is the DataFrame. A DataFrame holds the type of data you might think of as a table. This is similar to a sheet in Excel, or a table in a SQL database.

Pandas has powerful methods for most things you'll want to do with this type of data.

In [ ]:
df.describe().round(2)

The results show 8 numbers for each column in your original dataset. The first number, the count, shows how many rows have non-missing values.

Missing values arise for many reasons — sensors that fail to report, dropped measurements, or fields that simply don't apply to a given record. They are a common challenge in real-world data and often require cleaning or imputation before modelling. In this particular dataset, however, the count equals the total number of rows for every column, which confirms that there are no missing values to handle.

The second value is the **mean**, which is the average of all values in the column — a measure of the typical or central value. Below it, **std** is the standard deviation, which measures how widely the values are dispersed around that mean: a small std means the values cluster tightly near the average, while a large std indicates they are spread out over a broad range.

To interpret the **min**, **25%**, **50%**, **75%** and **max** values, imagine sorting each column from its lowest to its highest value. The first (smallest) entry is the **min**, and the last (largest) is the **max** — together they define the full range of the data. The percentiles then tell you how the values are distributed within that range. If you move a quarter of the way through the sorted list, you reach a value that is larger than 25% of the observations and smaller than the remaining 75%; this is the **25th percentile** (also called the *first quartile*). The **50th percentile** is the **median** — the middle value that splits the data into two equal halves — and the **75th percentile** (the *third quartile*) is larger than three-quarters of the values. Comparing these markers reveals the shape of the distribution: if the median sits far from the mean, or the gaps between quartiles are uneven, the data is skewed rather than symmetric.

### What we are looking for in these plots

Summary statistics describe each column in isolation; these four plots help us **see the data** and uncover the relationships that linear regression will try to capture. For each panel we have a specific question in mind:

- **Distribution of RTT** (top-left histogram) — What does the target variable look like? We check whether RTT is roughly bell-shaped or skewed, where most measurements concentrate, and whether there are outliers or a long tail. Linear regression works best when the target is reasonably well-behaved.
- **RTT vs Hop Count** (top-right scatter) — Is there a **linear trend** between the number of hops and latency? An upward, roughly straight band of points suggests that hops are a useful predictor and that a linear model is appropriate. The vertical spread hints at how much other factors also influence RTT.
- **RTT vs Geographic Distance** (bottom-left scatter) — Does latency grow with distance, as propagation delay would predict? A positive slope confirms our physical intuition and tells us distance is worth including as a feature.
- **Average RTT by Hour of Day** (bottom-right bar chart) — Does time of day matter? By averaging RTT per hour and shading the 9–18 business window, we look for a **congestion signal** — higher latency during peak hours — which would justify the `is_peak` feature.

In short, we are visually validating that our chosen features (hops, distance, peak hours) really do relate to RTT, and checking that those relationships are approximately linear before we fit a model.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Distribution of RTT
axes[0, 0].hist(df["rtt_ms"], bins=40, color="steelblue", edgecolor="white")
axes[0, 0].set_xlabel("RTT (ms)")
axes[0, 0].set_ylabel("Count")
axes[0, 0].set_title("Distribution of Round-Trip Time")

# RTT vs Hops
axes[0, 1].scatter(df["hops"], df["rtt_ms"], alpha=0.3, s=10, color="coral")
axes[0, 1].set_xlabel("Hop Count")
axes[0, 1].set_ylabel("RTT (ms)")
axes[0, 1].set_title("RTT vs Hop Count")

# RTT vs Distance
axes[1, 0].text(
    0.5, 0.5,
    "TODO (exercise):\nRTT vs Distance",
    transform=axes[1, 0].transAxes,
    ha="center", va="center",
    fontsize=12, color="seagreen",
    bbox=dict(boxstyle="round", facecolor="honeydew", edgecolor="seagreen"),
)
axes[1, 0].set_title("RTT vs Geographic Distance")

# RTT by time of day
hourly_rtt = df.groupby("hour")["rtt_ms"].mean()
axes[1, 1].bar(hourly_rtt.index, hourly_rtt.values, color="mediumpurple", edgecolor="white")
axes[1, 1].axvspan(9, 18, alpha=0.1, color="red", label="Peak hours")
axes[1, 1].set_xlabel("Hour of Day")
axes[1, 1].set_ylabel("Mean RTT (ms)")
axes[1, 1].set_title("Average RTT by Hour")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

### ✏️ Exercise: Complete the Missing Diagram

The figure above shows only **three** of the four planned panels. The bottom-left panel — **RTT vs Geographic Distance** — is missing. Your task is to recreate it.

**Exercise — Scatter plot of RTT vs Distance**

Using the same style as the "RTT vs Hop Count" scatter, draw a scatter plot of `rtt_ms` (y-axis) against `distance_km` (x-axis). This lets us check whether latency grows with geographic distance, as propagation delay would predict.

**Your goals:**
1. Draw a scatter plot of `df["distance_km"]` (x) vs `df["rtt_ms"]` (y) — use `alpha=0.3`, `s=10` and a colour such as `"seagreen"`.
2. Add a title and axis labels (x = "Distance (km)", y = "RTT (ms)").
3. Call `plt.show()` to display it.

**Questions to answer from your plot:**
- Does RTT tend to increase with distance?
- Is the relationship roughly linear, and how much vertical spread is there around the trend?

In [ ]:
# RTT vs Distance
# plt.scatter(df["distance_km"], df["rtt_ms"], alpha=0.3, s=10, color="seagreen")
# plt.xlabel("Distance (km)")
# plt.ylabel("RTT (ms)")
# plt.title("RTT vs Distance")
# plt.show()

### Correlation Matrix — Quantifying the Relationships

The EDA scatter plots let us *see* the relationships between features and RTT; a **correlation matrix** lets us *measure* them. `df.corr()` computes the **Pearson correlation coefficient** for every pair of columns — a number between −1 and +1 that captures the strength and direction of a *linear* relationship:

- **+1** → perfect positive linear relationship (as one rises, the other rises proportionally)
- **0** → no linear relationship
- **−1** → perfect negative linear relationship

This matters because **linear regression assumes linear relationships**. The matrix is a quick, numerical sanity check — *before fitting any model* — that our chosen features really do relate linearly to the target.

**Why the heatmap?** A table of numbers is hard to scan, so we colour-code it:
- `annot=True` prints the exact coefficient inside each cell.
- `cmap="coolwarm"` is a *diverging* palette (warm/red = positive, cool/blue = negative) — ideal because correlation has a natural midpoint of 0 with two opposite directions.
- `center=0` anchors the colour scale so 0 maps to neutral white; without it the colours would be misleading.
- `fmt=".3f"` shows three decimals — enough precision to compare features without clutter.

**Why the ranked list at the end?** It zooms in on the key question: *which features are most strongly correlated with RTT?* We select RTT's column, drop its self-correlation (always 1.0), and sort the rest. The ranking helps us **anticipate feature importance** — features at the top should carry the most predictive weight, while a feature with correlation near 0 (like raw `hour`) likely won't help on its own.

In [ ]:
# Correlation matrix
corr = df[["hops", "distance_km", "hour", "is_peak", "rtt_ms"]].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".3f")
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

print("\nCorrelation with RTT:")
print(corr["rtt_ms"].drop("rtt_ms").sort_values(ascending=False))

## 4. Train/Test Split

Before training, we separate **what we want to predict** from **what we predict it from**, and we set aside part of the data to test on.

- **Target `y`** — the column to predict, `rtt_ms`. Selecting one column with single brackets returns a 1-D **Series**, the shape scikit-learn expects for the target.
- **Features `X`** — the inputs the model learns from. We build *two* sets so we can compare them later: `X_hops_only` (just `hops`) and `X_multi` (`hops` + `distance_km` + `is_peak`). The **double brackets** `df[["hops"]]` return a 2-D **DataFrame** — scikit-learn always expects `X` to be 2-D (rows = samples, columns = features), even for a single feature.

**Why split?** We hold back data the model never sees during training so we can honestly measure how it performs on **new** inputs. Without this, a model could simply memorise the data and look perfect while failing in practice — a problem called **overfitting**.

`train_test_split` returns four parts in order — train inputs, test inputs, train target, test target:
- **`test_size=0.2`** — reserve 20% for testing, train on the other 80%.
- **`random_state=42`** — fixes the shuffling seed so the split is **reproducible** and identical every run.

Using the **same `random_state`** for both feature sets guarantees they share the exact same train/test rows, so any performance difference comes from the *features*, not from a lucky split. The `_, _` discard the duplicate target arrays (identical to `y_train`/`y_test`). The prints confirm roughly **800 training** and **200 test** samples.

In [ ]:
# Target
y = df["rtt_ms"]

# Features (we'll use different subsets for comparison)
X_hops_only = df[["hops"]]                             # Single feature
X_multi = df[["hops", "distance_km", "is_peak"]]       # Multi-feature

# Split (80% train, 20% test)
X_hops_train, X_hops_test, y_train, y_test = train_test_split(
    X_hops_only, y, test_size=0.2, random_state=42
)
X_multi_train, X_multi_test, _, _ = train_test_split(
    X_multi, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(y_train)}")
print(f"Test samples:     {len(y_test)}")

## 5. Model 1 — Single-Feature Linear Regression (Hops Only)

This cell **trains** the simplest possible model and reads off what it learned.

- `LinearRegression()` creates an *untrained* model — it knows the equation's form ($\text{RTT} = w \times \text{hops} + b$) but not yet the values of $w$ and $b$.
- `.fit(X_hops_train, y_train)` is where **learning** happens: scikit-learn finds the slope $w$ and intercept $b$ that minimise the sum of squared errors (the squared vertical gaps between the points and the line) — the classic **Ordinary Least Squares** fit. It trains only on the *training* set, never the test set.

After fitting, the model exposes what it learned (the trailing underscore marks attributes set during `.fit()`):
- **`coef_`** — the feature coefficients (slopes); with one feature we take `[0]` to get $w$.
- **`intercept_`** — the bias term $b$, the predicted RTT when `hops = 0`.

The power of linear regression is that these numbers are **directly interpretable** in networking terms:
- **`w` (slope ≈ 3.5 ms)** — the marginal cost of one extra hop: processing + queuing delay per router.
- **`b` (intercept)** — the **base latency** for a hypothetical 0-hop path: fixed overhead that absorbs the average effect of distance, peak hours, and noise this single-feature model can't yet see.

The `:.2f` simply formats each value to two decimals. This is our **baseline** — the multi-feature model in Section 6 will try to beat it.

In [ ]:
# Train single-feature model
model_single = LinearRegression()
model_single.fit(X_hops_train, y_train)

# Coefficients
w = model_single.coef_[0]
b = model_single.intercept_

print("=== Single-Feature Model (Hops → RTT) ===")
print(f"\nEquation: RTT = {w:.2f} × hops + {b:.2f}")
print(f"\nInterpretation:")
print(f"  • Each additional hop adds ~{w:.2f} ms to RTT")
print(f"  • Base latency (0 hops): ~{b:.2f} ms")

Training told us what the model learned; now we measure **how well it predicts on the held-out test set** — the 200 rows it never saw. `.predict(X_hops_test)` applies the learned equation to those inputs, and we compare the results against the true `y_test` using four complementary metrics:

- **MSE (Mean Squared Error)** — the average of the *squared* errors, $\frac{1}{n}\sum (y_i - \hat{y}_i)^2$. Squaring penalises large misses heavily, but its units (ms²) aren't intuitive.
- **RMSE (Root MSE)** — $\sqrt{\text{MSE}}$, which restores the original units (**ms**). Read it as the "typical" prediction error while still punishing big misses.
- **MAE (Mean Absolute Error)** — the average of the *absolute* errors, $\frac{1}{n}\sum |y_i - \hat{y}_i|$. No squaring, so it's **more robust to outliers** and reads as the average miss in ms. If **RMSE ≫ MAE**, a few large errors are inflating RMSE — a hint of outliers.
- **R² (coefficient of determination)** — the **fraction of RTT variance the model explains**, where 1.0 is perfect and 0 is no better than always predicting the mean (it can go negative). Being unitless, it's the natural score for **comparing models**.

The final line turns R² into a percentage (e.g. 0.62 → *"explains 62.0% of RTT variance"*). The remaining variance is what the multi-feature model in Section 6 aims to capture. We report all four because each adds something: RMSE/MAE give error in real units, their ratio flags outliers, and R² gives a normalised score for comparison.

In [ ]:
# Predictions and evaluation
y_pred_single = model_single.predict(X_hops_test)

mse_single = mean_squared_error(y_test, y_pred_single)
rmse_single = np.sqrt(mse_single)
mae_single = mean_absolute_error(y_test, y_pred_single)
r2_single = r2_score(y_test, y_pred_single)

print("=== Model 1 Evaluation (Test Set) ===")
print(f"  MSE:  {mse_single:.2f} ms²")
print(f"  RMSE: {rmse_single:.2f} ms")
print(f"  MAE:  {mae_single:.2f} ms")
print(f"  R²:   {r2_single:.4f}")
print(f"\n  → The model explains {r2_single*100:.1f}% of RTT variance using hops alone.")

Numbers alone are abstract — this plot lets us **see** the fit. We overlay two things on the same axes:

- **The actual test data** (blue points) — each point is one test measurement: its hop count (x) versus its true RTT (y). `alpha=0.4` makes overlapping points blend so we can see where measurements concentrate; `s=20` sets the marker size.
- **The learned regression line** (red) — the model's prediction across the hop range. We build it by generating 100 evenly spaced hop values with `np.linspace(3, 24, 100)`, reshaping them into a 2-D column with `.reshape(-1, 1)` (scikit-learn's `.predict()` requires 2-D input, and `-1` lets NumPy infer the row count), then calling `.predict()` to get the line's y-values. The `f`-string embeds the actual coefficients into the legend, so it reads e.g. *"Fit: RTT = 3.51×hops + 12.30"*.

`tight_layout()` prevents labels from overlapping and `show()` renders the figure. The key thing to look for is the **vertical spread** of points around the line: points scatter widely above and below it because hop count alone can't explain RTT — distance, peak hours, and noise also contribute. That visible gap is exactly what motivates the multi-feature model in Section 6.

In [ ]:
# Visualise: regression line
plt.figure(figsize=(10, 5))

plt.scatter(X_hops_test["hops"], y_test, alpha=0.4, s=20, label="Actual", color="steelblue")

# Plot regression line
x_line = pd.DataFrame({"hops": np.linspace(3, 24, 100)})
y_line = model_single.predict(x_line)
plt.plot(x_line["hops"], y_line, color="red", linewidth=2, label=f"Fit: RTT = {w:.2f}×hops + {b:.2f}")

plt.xlabel("Hop Count")
plt.ylabel("RTT (ms)")
plt.title("Single-Feature Linear Regression: Hops → RTT")
plt.legend()
plt.tight_layout()
plt.show()

### Observations on Model 1

The scatter shows significant **vertical spread** around the regression line — hops alone don't fully explain RTT. The remaining variance comes from:
- Geographic distance (propagation delay)
- Time of day (congestion)
- Random jitter

Let's add more features!

## 6. Model 2 — Multi-Feature Linear Regression

$$\hat{\text{RTT}} = w_1 \times \text{hops} + w_2 \times \text{distance\_km} + w_3 \times \text{is\_peak} + b$$

### ✏️ Exercise: Train the Multi-Feature Model

Model 1 used only `hops`. Now build **Model 2** using all three features (`hops`, `distance_km`, `is_peak`) and read off what it learned. The training set `X_multi_train` and target `y_train` were already prepared in Section 4.

**Your goals:**
1. Create a `LinearRegression()` and `.fit()` it on `X_multi_train` and `y_train`.
2. Collect the learned coefficients into a `dict` mapping each feature name to its coefficient (hint: `dict(zip(feature_names, model_multi.coef_))`).
3. Print the full equation and interpret each coefficient (per hop, per km, peak-hour penalty, base overhead via `intercept_`).

**Questions to answer:**
- How many ms does each additional hop add? Each 1000 km of distance?
- How much latency does measuring during business hours (`is_peak = 1`) add?
- Does the per-hop coefficient match the ~3.5 ms you found in Model 1?

In [ ]:
# Train multi-feature model
model_multi = LinearRegression()
# model_multi.fit()

# Coefficients
feature_names = X_multi.columns.tolist()
coefs = dict(zip(feature_names, model_multi.coef_))

print("=== Multi-Feature Model ===")
print(f"\nEquation: RTT = ", end="")
terms = [f"{coefs[f]:.4f}×{f}" for f in feature_names]
print(" + ".join(terms) + f" + {model_multi.intercept_:.2f}")

print(f"\nCoefficient Interpretation:")
print(f"  • hops:        +{coefs['hops']:.2f} ms per additional hop")
print(f"  • distance_km: +{coefs['distance_km']:.4f} ms per km (≈{coefs['distance_km']*1000:.2f} ms per 1000 km)")
print(f"  • is_peak:     +{coefs['is_peak']:.2f} ms during business hours")
print(f"  • intercept:   {model_multi.intercept_:.2f} ms (base overhead)")

### ✏️ Exercise: Evaluate Model 2 on the Test Set

You trained Model 2 above — now measure how well it predicts on the **held-out test set** (`X_multi_test`, `y_test`) and compare it to Model 1's scores. To avoid getting stuck, work through these **small, independent tasks one at a time** — each line stands on its own:

**Task 1 — Predict.** Call `model_multi.predict(X_multi_test)` and store the result in `y_pred_multi`.

**Task 2 — Compute the four metrics** (reuse the exact pattern from Model 1's evaluation):
- `mse_multi` ← `mean_squared_error(y_test, y_pred_multi)`
- `rmse_multi` ← `np.sqrt(mse_multi)`
- `mae_multi` ← `mean_absolute_error(y_test, y_pred_multi)`
- `r2_multi` ← `r2_score(y_test, y_pred_multi)`

**Task 3 — Print** each metric with units (MSE in ms², RMSE/MAE in ms, R² to 4 decimals).

**Task 4 — Report** the explained variance as a percentage: `r2_multi * 100`.

**Questions to answer:**
- Is `r2_multi` higher than `r2_single`? By how much?
- Did RMSE drop compared to Model 1? What does that mean in milliseconds?

In [ ]:
# Task 1 — Predictions on the test set
y_pred_multi = model_multi.predict(X_multi_test)

# Task 2 — Compute evaluation metrics
# mse_multi  = mean_squared_error()
# rmse_multi = np.sqrt()
# mae_multi  = mean_absolute_error()
# r2_multi   = r2_score()

# Task 3 — Print the metrics with units
print("=== Model 2 Evaluation (Test Set) ===")
print(f"  MSE:  {mse_multi:.2f} ms²")
print(f"  RMSE: {rmse_multi:.2f} ms")
print(f"  MAE:  {mae_multi:.2f} ms")
print(f"  R²:   {r2_multi:.4f}")

# Task 4 — Report explained variance as a percentage
print(f"\n  → The model explains {r2_multi*100:.1f}% of RTT variance.")

## 7. Model Comparison

In [ ]:
# Comparison table
comparison = pd.DataFrame({
    "Model": ["Single (hops only)", "Multi (hops + distance + peak)"],
    "MSE (ms²)": [mse_single, mse_multi],
    "RMSE (ms)": [rmse_single, rmse_multi],
    "MAE (ms)": [mae_single, mae_multi],
    "R²": [r2_single, r2_multi],
}).round(4)

print("=== Model Comparison ===")
print(comparison.to_string(index=False))

improvement = (1 - mse_multi / mse_single) * 100
print(f"\n→ Multi-feature model reduces MSE by {improvement:.1f}%")
print(f"→ R² improved from {r2_single:.3f} to {r2_multi:.3f}")

In [ ]:
# Visual comparison: Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Model 1
axes[0].scatter(y_test, y_pred_single, alpha=0.3, s=10, color="coral")
axes[0].plot([0, 150], [0, 150], "k--", linewidth=1, label="Perfect prediction")
axes[0].set_xlabel("Actual RTT (ms)")
axes[0].set_ylabel("Predicted RTT (ms)")
axes[0].set_title(f"Model 1: Hops Only (R²={r2_single:.3f})")
axes[0].legend()

# Model 2
axes[1].scatter(y_test, y_pred_multi, alpha=0.3, s=10, color="seagreen")
axes[1].plot([0, 150], [0, 150], "k--", linewidth=1, label="Perfect prediction")
axes[1].set_xlabel("Actual RTT (ms)")
axes[1].set_ylabel("Predicted RTT (ms)")
axes[1].set_title(f"Model 2: Multi-Feature (R²={r2_multi:.3f})")
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Residual Analysis

Residuals = Actual − Predicted. A good model should have residuals:
- Centred around 0
- Randomly scattered (no patterns)
- Approximately normally distributed

In [ ]:
residuals_single = y_test - y_pred_single
residuals_multi = y_test - y_pred_multi

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Residuals vs predicted (Model 1)
axes[0, 0].scatter(y_pred_single, residuals_single, alpha=0.3, s=10, color="coral")
axes[0, 0].axhline(y=0, color="black", linestyle="--")
axes[0, 0].set_xlabel("Predicted RTT (ms)")
axes[0, 0].set_ylabel("Residual (ms)")
axes[0, 0].set_title("Model 1: Residuals vs Predicted")

# Residuals vs predicted (Model 2)
axes[0, 1].scatter(y_pred_multi, residuals_multi, alpha=0.3, s=10, color="seagreen")
axes[0, 1].axhline(y=0, color="black", linestyle="--")
axes[0, 1].set_xlabel("Predicted RTT (ms)")
axes[0, 1].set_ylabel("Residual (ms)")
axes[0, 1].set_title("Model 2: Residuals vs Predicted")

# Residual distributions
axes[1, 0].hist(residuals_single, bins=40, color="coral", edgecolor="white")
axes[1, 0].axvline(x=0, color="black", linestyle="--")
axes[1, 0].set_xlabel("Residual (ms)")
axes[1, 0].set_title(f"Model 1: Residual Distribution (std={residuals_single.std():.2f})")

axes[1, 1].hist(residuals_multi, bins=40, color="seagreen", edgecolor="white")
axes[1, 1].axvline(x=0, color="black", linestyle="--")
axes[1, 1].set_xlabel("Residual (ms)")
axes[1, 1].set_title(f"Model 2: Residual Distribution (std={residuals_multi.std():.2f})")

plt.tight_layout()
plt.show()

## 9. Cross-Validation

In [ ]:
# 5-fold cross-validation
cv_single = cross_val_score(LinearRegression(), X_hops_only, y, cv=5, scoring="r2")
cv_multi = cross_val_score(LinearRegression(), X_multi, y, cv=5, scoring="r2")

print("=== 5-Fold Cross-Validation (R² scores) ===")
print(f"\nModel 1 (hops only):  {cv_single.mean():.4f} ± {cv_single.std():.4f}")
print(f"  Folds: {[f'{s:.4f}' for s in cv_single]}")
print(f"\nModel 2 (multi):      {cv_multi.mean():.4f} ± {cv_multi.std():.4f}")
print(f"  Folds: {[f'{s:.4f}' for s in cv_multi]}")
print(f"\n→ Cross-validation confirms Model 2 is consistently better.")

## 10. Feature Importance via Coefficient Magnitude

To compare coefficients fairly, we need to standardise features (mean=0, std=1).

In [ ]:
# Standardised coefficients for fair comparison
scaler = StandardScaler()
X_multi_scaled = scaler.fit_transform(X_multi)

model_scaled = LinearRegression()
model_scaled.fit(X_multi_scaled, y)

importance = pd.DataFrame({
    "Feature": feature_names,
    "Raw Coefficient": model_multi.coef_,
    "Standardised Coefficient": model_scaled.coef_,
    "|Standardised|": np.abs(model_scaled.coef_),
}).sort_values("|Standardised|", ascending=False)

print("=== Feature Importance (Standardised Coefficients) ===")
print(importance.to_string(index=False))

# Bar chart
plt.figure(figsize=(8, 4))
colors = ["steelblue", "seagreen", "mediumpurple"]
plt.barh(importance["Feature"], importance["|Standardised|"], color=colors)
plt.xlabel("Absolute Standardised Coefficient")
plt.title("Feature Importance for RTT Prediction")
plt.tight_layout()
plt.show()

## 11. Practical Predictions

Use the model to predict RTT for specific network paths.

In [ ]:
# Predict RTT for realistic scenarios
scenarios = pd.DataFrame({
    "Scenario": [
        "Local data centre (same city)",
        "Regional (Paris → London)",
        "Continental (Paris → Moscow)",
        "Intercontinental (Paris → Tokyo)",
        "Same region, peak hours",
        "Same region, off-peak",
    ],
    "hops": [4, 8, 14, 20, 8, 8],
    "distance_km": [50, 350, 2500, 9700, 350, 350],
    "is_peak": [0, 0, 0, 0, 1, 0],
})

X_scenarios = scenarios[["hops", "distance_km", "is_peak"]]
scenarios["Predicted RTT (ms)"] = model_multi.predict(X_scenarios).round(1)

print("=== RTT Predictions for Network Paths ===")
print(scenarios[["Scenario", "hops", "distance_km", "is_peak", "Predicted RTT (ms)"]].to_string(index=False))

## 12. Limitations of Linear Regression for Network Latency

Let's explore where the linear model breaks down.

In [ ]:
# Simulate non-linear behaviour: congestion spikes at high utilisation
# Real networks show exponential latency growth when utilisation > 80%

utilisation = np.linspace(0, 0.99, 200)
# M/M/1 queuing model: delay ~ 1/(1 - utilisation)
queuing_delay = 5 / (1 - utilisation)

plt.figure(figsize=(10, 5))
plt.plot(utilisation * 100, queuing_delay, color="coral", linewidth=2, label="Actual (M/M/1 queue)")

# What linear regression would predict
from numpy.polynomial import polynomial as P
linear_fit = np.polyfit(utilisation[:150], queuing_delay[:150], 1)  # fit on 0-75%
linear_pred = np.polyval(linear_fit, utilisation)
plt.plot(utilisation * 100, linear_pred, "--", color="steelblue", linewidth=2, label="Linear approximation")

plt.axvline(x=80, color="red", linestyle=":", alpha=0.7, label="80% threshold")
plt.xlabel("Link Utilisation (%)")
plt.ylabel("Queuing Delay (ms)")
plt.title("Why Linear Regression Fails at High Utilisation")
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()

print("Key Insight: Network latency becomes NON-LINEAR when links approach saturation.")
print("Linear regression works well for normal conditions but fails at extremes.")
print("\nPossible improvements:")
print("  • Add polynomial features (hops², distance²)")
print("  • Use a non-linear model (decision tree, random forest)")
print("  • Segment by utilisation level and fit separate models")

## 13. Summary & Key Takeaways

| Aspect | Finding |
|:-------|:--------|
| **Single-feature** | Hops alone explain part of RTT variance |
| **Multi-feature** | Adding distance and peak-hour improves R² significantly |
| **Coefficients** | Each hop ≈ 3–4 ms, each 1000 km ≈ 5 ms, peak hours ≈ +20 ms |
| **Residuals** | Well-behaved (centred, normal) for the multi-feature model |
| **Limitations** | Linear model fails at high utilisation (exponential queuing) |

### Physical Interpretation

- **Per-hop delay** ($w_{\text{hops}}$ ≈ 3.5 ms): processing + queuing at each router
- **Propagation** ($w_{\text{distance}}$ ≈ 0.005 ms/km): speed of light in fibre (2/3 c)
- **Congestion** ($w_{\text{peak}}$ ≈ 20 ms): buffer bloat during business hours